# Group-Aware Dataset Partitioning

This notebook documents the original train, validation, and locked-test partitioning procedure. It uses Actual Place Name as an exact-case business-place grouping rule, performs stratified group-aware splits, and records the resulting partition and OOF-fold metadata.


## Release Boundary

This is a restricted code-inspection copy. It retains the original analytical logic, but it does not include Banjarbaru source data, review-level records, manual-audit files, trained checkpoints, or intermediate outputs. The paths in the code refer to the authorized analysis environment and are not expected to run from this public package.


### Import partitioning dependencies and configure locations

Imports the split utilities, fixes random state 42, and defines the historical processed-data, split, locked-test, and figure directories.


In [ ]:
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
from collections import Counter

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42  # Untuk reproducibility
np.random.seed(RANDOM_STATE)

BASE_DIR = os.path.dirname(os.getcwd())
DATA_PROCESSED = os.path.join(BASE_DIR, 'data', 'processed')
DATA_SPLITS = os.path.join(BASE_DIR, 'data', 'splits')
DATA_FINAL_TEST = os.path.join(BASE_DIR, 'data', 'final_test_locked')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')

os.makedirs(DATA_SPLITS, exist_ok=True)
os.makedirs(DATA_FINAL_TEST, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("="*60)
print("SETUP KONFIGURASI")
print("="*60)
print(f"Random State: {RANDOM_STATE}")
print(f"Data Processed: {DATA_PROCESSED}")
print(f"Data Splits Output: {DATA_SPLITS}")
print(f"Final Test Locked: {DATA_FINAL_TEST}")
print(f"Figures Directory: {FIGURES_DIR}")


### Load the processed silver-labelled dataset

Reads the restricted processed review dataset and reports the available columns before splitting.


In [ ]:
input_file = os.path.join(DATA_PROCESSED, 'preprocessed_reviews.csv')
df = pd.read_csv(input_file)

print("="*60)
print("INFORMASI DATASET")
print("="*60)
print(f"Total sampel: {len(df):,}")
print(f"Total kolom: {len(df.columns)}")
print(f"\nKolom yang tersedia:")
for col in df.columns:
    print(f"  - {col}")


### Inspect required modelling fields locally

Displays a local preview of the group, place-type, cleaned-text, and silver-label fields needed for partitioning. No review rows are released.


In [ ]:
print("\nSampel data (5 baris pertama):")
df[['Actual Place Name', 'Place Type', 'text_clean', 'sentiment_label']].head()


### Summarize corpus-level silver-label balance

Calculates the overall class counts, percentages, and imbalance ratio that the stratified split attempts to preserve.


In [ ]:
print("="*60)
print("DISTRIBUSI KELAS SENTIMEN (KESELURUHAN)")
print("="*60)

sentiment_counts = df['sentiment_label'].value_counts()
sentiment_pct = df['sentiment_label'].value_counts(normalize=True) * 100

print("\nDistribusi kelas:")
for label in ['positif', 'netral', 'negatif']:
    if label in sentiment_counts.index:
        print(f"  {label.capitalize():10}: {sentiment_counts[label]:,} ({sentiment_pct[label]:.2f}%)")

max_class = sentiment_counts.max()
min_class = sentiment_counts.min()
imbalance_ratio = max_class / min_class
print(f"\nRasio imbalance (max/min): {imbalance_ratio:.2f}:1")


### Exact-case business-place groups

Counts unique Actual Place Name values and summarizes the number of reviews associated with each exact-case group.


In [ ]:
print("="*60)
print("ANALISIS GROUP (TEMPAT USAHA)")
print("="*60)

n_unique_places = df['Actual Place Name'].nunique()
reviews_per_place = df.groupby('Actual Place Name').size()

print(f"\nTotal tempat usaha unik: {n_unique_places:,}")
print(f"\nStatistik ulasan per tempat usaha:")
print(f"  Min    : {reviews_per_place.min()}")
print(f"  Max    : {reviews_per_place.max()}")
print(f"  Mean   : {reviews_per_place.mean():.2f}")
print(f"  Median : {reviews_per_place.median():.2f}")

print(f"\nDistribusi tempat usaha berdasarkan jumlah ulasan:")
bins = [1, 2, 5, 10, 20, 50, 100, float('inf')]
labels = ['1', '2-4', '5-9', '10-19', '20-49', '50-99', '100+']
place_dist = pd.cut(reviews_per_place, bins=bins, labels=labels, right=False).value_counts().sort_index()
for label, count in place_dist.items():
    print(f"  {label:10} ulasan: {count:,} tempat")


### Prepare sample, label, and group arrays

Extracts row indices, LSTM-derived silver labels, and Actual Place Name group values for the split functions.


In [ ]:
X = df.index.values 
y = df['sentiment_label'].values 
groups = df['Actual Place Name'].values 

print("="*60)
print("PERSIAPAN SPLITTING")
print("="*60)
print(f"Total sampel (X): {len(X):,}")
print(f"Total label (y): {len(y):,}")
print(f"Total group unik: {len(np.unique(groups)):,}")


### Define the stratified group-split helper

Creates the helper that uses StratifiedGroupKFold to produce group-aware partitions at the requested approximate holdout proportion.


In [ ]:
def stratified_group_split(X, y, groups, test_size=0.15, random_state=42):
    n_splits = int(1 / test_size)
    
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for train_idx, test_idx in sgkf.split(X, y, groups):
        return train_idx, test_idx

print("\nFungsi stratified_group_split berhasil didefinisikan.")


### Create the locked-test partition

Separates the first train-plus-validation set from the locked-test set and checks overlap of exact-case business-place groups.


In [ ]:
print("="*60)
print("TAHAP 1: SPLIT TEST SET (LOCKED)")
print("="*60)

trainval_idx, test_idx = stratified_group_split(X, y, groups, test_size=0.15, random_state=RANDOM_STATE)

trainval_groups = set(groups[trainval_idx])
test_groups = set(groups[test_idx])
leaked_groups = trainval_groups.intersection(test_groups)

print(f"\nHasil Split Tahap 1:")
print(f"  Train+Val: {len(trainval_idx):,} sampel ({len(trainval_idx)/len(X)*100:.1f}%)")
print(f"  Test     : {len(test_idx):,} sampel ({len(test_idx)/len(X)*100:.1f}%)")
print(f"\nVerifikasi Kebocoran Group:")
print(f"  Group di Train+Val: {len(trainval_groups):,}")
print(f"  Group di Test     : {len(test_groups):,}")
print(f"  Group yang bocor  : {len(leaked_groups)}")

if len(leaked_groups) == 0:
    print("TIDAK ADA KEBOCORAN GROUP!")
else:
    print(f"PERINGATAN: Ada {len(leaked_groups)} group yang bocor!")


### Create the train and validation partitions

Splits the train-plus-validation portion again, then checks all three pairwise group-overlap conditions under the exact-case grouping rule.


In [ ]:
print("="*60)
print("TAHAP 2: SPLIT VALIDATION SET")
print("="*60)

X_trainval = X[trainval_idx]
y_trainval = y[trainval_idx]
groups_trainval = groups[trainval_idx]

train_idx_local, val_idx_local = stratified_group_split(
    X_trainval, y_trainval, groups_trainval, 
    test_size=0.176, random_state=RANDOM_STATE
)

train_idx = trainval_idx[train_idx_local]
val_idx = trainval_idx[val_idx_local]

train_groups = set(groups[train_idx])
val_groups = set(groups[val_idx])
leaked_train_val = train_groups.intersection(val_groups)
leaked_train_test = train_groups.intersection(test_groups)
leaked_val_test = val_groups.intersection(test_groups)

print(f"\nHasil Split Final:")
print(f"  Train     : {len(train_idx):,} sampel ({len(train_idx)/len(X)*100:.1f}%)")
print(f"  Validation: {len(val_idx):,} sampel ({len(val_idx)/len(X)*100:.1f}%)")
print(f"  Test      : {len(test_idx):,} sampel ({len(test_idx)/len(X)*100:.1f}%)")

print(f"\nVerifikasi Kebocoran Group:")
print(f"  Train Val  : {len(leaked_train_val)} group")
print(f"  Train Test : {len(leaked_train_test)} group")
print(f"  Val Test   : {len(leaked_val_test)} group")

total_leaked = len(leaked_train_val) + len(leaked_train_test) + len(leaked_val_test)
if total_leaked == 0:
    print("\nSEMUA SUBSET BEBAS DARI KEBOCORAN GROUP!")
else:
    print(f"\nPERINGATAN: Total {total_leaked} kebocoran terdeteksi!")


### Materialize the three partition data frames

Creates train, validation, and locked-test data frames and marks each record with its partition role.


In [ ]:
df_train = df.iloc[train_idx].copy().reset_index(drop=True)
df_val = df.iloc[val_idx].copy().reset_index(drop=True)
df_test = df.iloc[test_idx].copy().reset_index(drop=True)

df_train['split_type'] = 'train'
df_val['split_type'] = 'validation'
df_test['split_type'] = 'test'

print("="*60)
print("DATAFRAME SETIAP SUBSET")
print("="*60)
print(f"\ndf_train : {len(df_train):,} baris")
print(f"df_val   : {len(df_val):,} baris")
print(f"df_test  : {len(df_test):,} baris")
print(f"Total    : {len(df_train) + len(df_val) + len(df_test):,} baris")


### Verify class balance within each partition

Reports silver-label counts, percentages, and unique exact-case group counts for train, validation, and locked-test partitions.


In [ ]:
def print_distribution(df_subset, name):

    counts = df_subset['sentiment_label'].value_counts()
    pcts = df_subset['sentiment_label'].value_counts(normalize=True) * 100
    
    print(f"\n{name}:")
    print(f"  {'Kelas':<10} {'Jumlah':>8} {'Persentase':>12}")
    print(f"  {'-'*32}")
    for label in ['positif', 'netral', 'negatif']:
        if label in counts.index:
            print(f"  {label.capitalize():<10} {counts[label]:>8,} {pcts[label]:>11.2f}%")
    
    n_places = df_subset['Actual Place Name'].nunique()
    print(f"  {'-'*32}")
    print(f"  Tempat usaha unik: {n_places:,}")
    
    return counts, pcts

print("="*60)
print("VERIFIKASI DISTRIBUSI KELAS PER SUBSET")
print("="*60)

train_counts, train_pcts = print_distribution(df_train, "TRAIN")
val_counts, val_pcts = print_distribution(df_val, "VALIDATION")
test_counts, test_pcts = print_distribution(df_test, "TEST")


### Persist restricted train, validation, and locked-test files

Writes the historical partition CSV files, including the label-bearing locked-test file retained only in the authorized environment.


In [ ]:
print("="*60)
print("MENYIMPAN DATASET SPLIT")
print("="*60)

train_file = os.path.join(DATA_SPLITS, 'train.csv')
val_file = os.path.join(DATA_SPLITS, 'validation.csv')

df_train.to_csv(train_file, index=False)
df_val.to_csv(val_file, index=False)

print(f"\n“Train set disimpan: {train_file}")
print(f"  Ukuran: {len(df_train):,} baris")

print(f"\nValidation set disimpan: {val_file}")
print(f"  Ukuran: {len(df_val):,} baris")

test_file = os.path.join(DATA_FINAL_TEST, 'test_locked.csv')
df_test.to_csv(test_file, index=False)

print(f"\n“Test set (LOCKED) disimpan: {test_file}")
print(f"  Ukuran: {len(df_test):,} baris")
print(f"\nPERINGATAN: Test set HANYA digunakan untuk evaluasi AKHIR!")


### Write split metadata

Creates split_metadata.json with sample counts, exact-case group counts, class distributions, overlap checks, and the fixed random state.


In [ ]:
split_metadata = {
    'timestamp': datetime.now().isoformat(),
    'random_state': RANDOM_STATE,
    'source_file': 'preprocessed_reviews.csv',
    'total_samples': len(df),
    'total_unique_places': int(df['Actual Place Name'].nunique()),
    'group_column': 'Actual Place Name',
    'label_column': 'sentiment_label',
    'splits': {
        'train': {
            'n_samples': len(df_train),
            'percentage': round(len(df_train) / len(df) * 100, 2),
            'n_unique_places': int(df_train['Actual Place Name'].nunique()),
            'class_distribution': {
                'positif': int(train_counts.get('positif', 0)),
                'netral': int(train_counts.get('netral', 0)),
                'negatif': int(train_counts.get('negatif', 0))
            },
            'class_percentage': {
                'positif': round(train_pcts.get('positif', 0), 2),
                'netral': round(train_pcts.get('netral', 0), 2),
                'negatif': round(train_pcts.get('negatif', 0), 2)
            },
            'file': 'splits/train.csv'
        },
        'validation': {
            'n_samples': len(df_val),
            'percentage': round(len(df_val) / len(df) * 100, 2),
            'n_unique_places': int(df_val['Actual Place Name'].nunique()),
            'class_distribution': {
                'positif': int(val_counts.get('positif', 0)),
                'netral': int(val_counts.get('netral', 0)),
                'negatif': int(val_counts.get('negatif', 0))
            },
            'class_percentage': {
                'positif': round(val_pcts.get('positif', 0), 2),
                'netral': round(val_pcts.get('netral', 0), 2),
                'negatif': round(val_pcts.get('negatif', 0), 2)
            },
            'file': 'splits/validation.csv'
        },
        'test': {
            'n_samples': len(df_test),
            'percentage': round(len(df_test) / len(df) * 100, 2),
            'n_unique_places': int(df_test['Actual Place Name'].nunique()),
            'class_distribution': {
                'positif': int(test_counts.get('positif', 0)),
                'netral': int(test_counts.get('netral', 0)),
                'negatif': int(test_counts.get('negatif', 0))
            },
            'class_percentage': {
                'positif': round(test_pcts.get('positif', 0), 2),
                'netral': round(test_pcts.get('netral', 0), 2),
                'negatif': round(test_pcts.get('negatif', 0), 2)
            },
            'file': 'final_test_locked/test_locked.csv',
            'locked': True,
            'note': 'HANYA untuk evaluasi akhir, jangan gunakan untuk tuning!'
        }
    },
    'leakage_check': {
        'train_val_overlap': 0,
        'train_test_overlap': 0,
        'val_test_overlap': 0,
        'status': 'PASSED - No group leakage detected'
    },
    'stratification_check': {
        'max_deviation_from_original': round(max(
            abs(train_pcts.get('positif', 0) - sentiment_pct.get('positif', 0)),
            abs(train_pcts.get('netral', 0) - sentiment_pct.get('netral', 0)),
            abs(train_pcts.get('negatif', 0) - sentiment_pct.get('negatif', 0)),
            abs(val_pcts.get('positif', 0) - sentiment_pct.get('positif', 0)),
            abs(val_pcts.get('netral', 0) - sentiment_pct.get('netral', 0)),
            abs(val_pcts.get('negatif', 0) - sentiment_pct.get('negatif', 0)),
            abs(test_pcts.get('positif', 0) - sentiment_pct.get('positif', 0)),
            abs(test_pcts.get('netral', 0) - sentiment_pct.get('netral', 0)),
            abs(test_pcts.get('negatif', 0) - sentiment_pct.get('negatif', 0))
        ), 2),
        'status': 'Good' if max(
            abs(train_pcts.get('positif', 0) - sentiment_pct.get('positif', 0)),
            abs(val_pcts.get('positif', 0) - sentiment_pct.get('positif', 0)),
            abs(test_pcts.get('positif', 0) - sentiment_pct.get('positif', 0))
        ) < 5 else 'Warning - deviation > 5%'
    }
}

metadata_file = os.path.join(DATA_SPLITS, 'split_metadata.json')
with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(split_metadata, f, indent=2, ensure_ascii=False)

print("="*60)
print("METADATA SPLIT")
print("="*60)
print(f"\n“ Metadata disimpan: {metadata_file}")
print(f"\nIsi metadata:")
print(json.dumps(split_metadata, indent=2, ensure_ascii=False))
